# From Trend to Waste: every algorithm, run step by step

*Business Data Analytics project · Madhusudhanan G · Pharos Sophy Samuel T J · Sri Charan*

This notebook reruns the project's analysis in Google Colab. Each step has three parts:
1. **What it does**, in plain words.
2. **The algorithm**: the actual code from our project (`src/`).
3. **Run it and see the results**: the same numbers and charts as in our slides.

**How to run it (about 10-15 minutes):**
1. Put `From_Trend_to_Waste_colab_pack.zip` in your Google Drive (anywhere in *My Drive*), or upload it when the first cell asks.
2. Click **Runtime → Run all**, and allow Drive access when asked.

All retail data comes from **one retailer, Nuna Lie** (an Italian fast-fashion chain), through the **VISUELLE 2.0** dataset: sales, deliveries, prices, discounts and loyalty-card purchases, linked by product code, shop and date. The consumer side is our own survey.

*The pack contains licensed data (VISUELLE 2.0) and our survey export. Use it for this course only and do not share it publicly.*

| Step | Question | Algorithms |
|---|---|---|
| 1 | How much stock is left over? Can we forecast the season after 2 weeks? What if we act early? | Stock replay, naive forecast, Poisson regression, gradient boosting, WAPE, fixed-effects regression, simulation, break-even |
| 2 | How do products sell over their life? When are they marked down? | K-means clustering, silhouette score, short-lived rule, markdown detection |
| 3 | Can we predict high leftover before launch, and after 2 weeks? | Logistic regression, decision tree, gradient boosting, ROC-AUC, odds ratios |
| 4 | Who buys? | RFM features, K-means segmentation, CLTV, Spearman correlation |
| 5 | What do market trends show? | Moving average, linear trendline, seasonal index, Google Trends test |
| 6 | How would we prove it works in real shops? | Power analysis (t-test) |
| 7 | What does excess stock cost? | Financial statements built from the data (markdown rate, stock-to-sales, turnover, margins) |
| 8 | What do consumers say? | Conjoint analysis (OLS with fixed effects), bootstrap, K-means, Spearman |

## 0 · Setup

In [ ]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = "google.colab" in sys.modules
PACK_NAME = "From_Trend_to_Waste_colab_pack.zip"
WORK = "/content/BDA-Project" if IN_COLAB else os.path.abspath("bda_colab_run")

if IN_COLAB:
    from google.colab import drive
    hits = []
    try:
        drive.mount("/content/drive")
        hits = (glob.glob(f"/content/drive/MyDrive/{PACK_NAME}") + glob.glob(f"/content/drive/MyDrive/*/{PACK_NAME}")
                or glob.glob(f"/content/drive/MyDrive/**/{PACK_NAME}", recursive=True))
    except Exception as e:
        print("Drive not mounted:", e)
    if not hits:
        from google.colab import files
        print(f"{PACK_NAME} not found in Drive: please choose it to upload")
        hits = [os.path.abspath(name) for name in files.upload()]
    pack = hits[0]
else:  # running on a laptop: the pack sits next to the project in deliverables/colab/
    pack = os.environ.get("BDA_PACK", os.path.join("..", "deliverables", "colab", PACK_NAME))

os.makedirs(WORK, exist_ok=True)
zipfile.ZipFile(pack).extractall(WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("Unpacked", os.path.basename(pack), "into", WORK)

In [ ]:
# Same library versions as the project (falls back to Colab's own versions if a pin is unavailable)
def pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode

if IN_COLAB and pip("scikit-learn==1.9.1", "statsmodels==0.15.0", "polars==1.44.2", "duckdb==1.5.5") != 0:
    pip("-U", "scikit-learn", "statsmodels", "polars", "duckdb")

In [ ]:
import inspect, json, warnings
import pandas as pd
from IPython.display import Code, Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


def show(fn, start=None, stop=None):
    """Print the project's own code for a function (optionally only the lines from `start` to `stop`)."""
    lines = inspect.getsource(fn).splitlines()
    i = next((k for k, l in enumerate(lines) if start in l), 0) if start else 0
    j = next((k for k, l in enumerate(lines) if k >= i and stop in l), len(lines) - 1) if stop else len(lines) - 1
    display(Code("\n".join(lines[i:j + 1]), language="python"))


def fig(name, width=900):
    display(Image(filename=f"outputs/figures/{name}.png", width=width))


def table(name):
    return pd.read_csv(f"outputs/tables/{name}.csv")


def js(name):
    return json.load(open(f"outputs/tables/{name}.json"))


def say(text):
    display(Markdown(text))


pct = lambda x, d=1: f"{x * 100:.{d}f}%"

## 1 · Leftover stock, week-2 forecasting and the what-if simulation

**What it does.** VISUELLE 2.0 has real deliveries and weekly sales for 5,355 products in 110 shops (106,850 product-shops).
- **Leftover stock**: deliveries minus sales after each product's first 12 weeks, replayed shop by shop, week by week.
- **Forecasting weeks 3-12 from weeks 1-2**: a **naive** forecast (last week repeats), a **category lifecycle profile**, **Poisson regression** and **gradient boosting**, with and without **Google Trends**. Trained on the 2017-18 seasons, tested on 2019. Error is measured with **WAPE** (total absolute error ÷ total sales).
- **Markdown uplift**: a **fixed-effects regression** estimates how much a discount raises sales.
- **Simulation**: replay the 2019 seasons. Products the forecast flags at week 2 (predicted sell-through below 60%) get fewer later deliveries and/or a 20% markdown from week 4. We compare unsold stock, units sold and revenue.

**The algorithm (our code):**

In [ ]:
from src import vis_forecast as V
show(V.stock_path)          # stock at the end of each week = deliveries so far - sales so far
show(V.fit_forecasters)     # naive, lifecycle profile, Poisson regression, gradient boosting (+/- Google Trends)
show(V.forecast_metrics)    # WAPE
show(V.estimate_uplift)     # fixed-effects regression for the markdown uplift
show(V.simulate)            # week-by-week stock replay under a policy

In [ ]:
V.main()   # run step 1 (the longest step, a few minutes)

In [ ]:
f = table("vis_forecast_metrics")
display(f[f.group == "All products"][["model", "weekly_wape", "remaining_total_wape", "bias"]])
v = js("vis_summary")
st, sim = v["stock"], v["simulation_main_scenario"]
say(f"""**Results**
- **{pct(st["leftover_share_of_delivered"])} of delivered units were still unsold after 12 weeks**; only {pct(st["share_shop_weeks_ending_at_zero_stock"])} of shop-weeks ended at zero stock
- Best forecast: **{v["forecast_best_model"]}**, weekly WAPE **{pct(v["forecast_best_weekly_wape"], 0)}** vs naive {pct(v["forecast_naive_weekly_wape"], 0)}
- Week-2 rule (flag if predicted sell-through < 60%, cut later deliveries 30%, 20% markdown from week 4): unsold stock **{pct(sim["leftover_change"])}**, units sold {pct(sim["sold_change"])}, revenue **{pct(sim["revenue_change"])}**""")
fig("vis_leftover_by_season", 600); fig("vis_forecast_models"); fig("vis_forecast_examples"); fig("vis_simulation")

## 2 · Demand lifecycles and markdown timing (end of season)

**What it does.** For every product with at least 30 units sold, we add up its first 12 weeks of sales across all shops and turn it into a curve showing the share of its sales in each week. Then:
- **K-means clustering** groups products with similar curves. The **silhouette score** helps choose how many groups to use.
- A **fixed rule** marks a product *short-lived* if its weekly sales fall below a quarter of the peak within 4 weeks of the peak, **and** weeks 7-12 carry less than 20% of its sales.
- **Markdown detection**: a product counts as marked down in any week its average discount across shops is at least 15%.

**The algorithm (our code):**

In [ ]:
from src import vis_lifecycle as LC
show(V.short_lived_flag)                                          # the short-lived rule
show(LC.main, "for k in range(3, 8):", "names = name_clusters")   # K-means for k = 3..7 + silhouette score
show(LC.name_clusters)                                            # readable names from each cluster's shape

In [ ]:
LC.main()   # run step 2

In [ ]:
s = js("vis_lifecycle_summary")
am = s["all_products_markdown"]
say(f"""**Results**
- Silhouette by number of clusters: {", ".join(f"k={k}: {v:.3f}" for k, v in s["silhouette_by_k"].items())} → we use **k = {s["k_chosen"]}**
- Clusters: {", ".join(f"{n} {pct(v, 0)}" for n, v in s["cluster_shares"].items())}
- Short-lived products: **{pct(s["share_short_lived"])}** ({s["short_lived_products"]} of {s["products_in_sample"]:,}); median sell-through {pct(s["short_lived_sell_through_median"], 0)} vs {pct(s["other_sell_through_median"], 0)} for the others
- **{pct(am["marked_down_after_demand_halved"], 0)} of marked-down products were cut only after demand had halved**; demand at the first markdown was {pct(am["median_demand_at_markdown_vs_peak"], 0)} of peak (median)""")
fig("vis_cluster_silhouette", 500); fig("vis_lifecycle_clusters"); fig("vis_markdown_vs_demand")

## 3 · Predicting high leftover (before launch → after 2 weeks)

**What it does.** A product has *high leftover* if less than 60% of its stock sold within 12 weeks. We train on the 2017-18 seasons and test on the 2019 seasons, which the models never saw, at three stages:
- **Product only**: category, colour, fabric, season, price position in its category, release month.
- **+ buying plan**: number of shops and initial units per shop.
- **After 2 weeks**: plus the first two weeks of sales.

**Logistic regression**, a **decision tree** and **gradient boosting** each give the probability of high leftover. **ROC-AUC** measures how well they rank risky products (0.5 = guessing). **Odds ratios** (statsmodels logit) show which features raise or lower the risk.

**The algorithm (our code):**

In [ ]:
from src import vis_risk_model as R
show(R.main, "models = {", "scores[(stage, name)] = s")   # three models at three stages
show(R.evaluate)
show(R.main, "formula = (", "lm = smf.logit")              # odds-ratio model

In [ ]:
R.main()   # run step 3

In [ ]:
m = table("vis_risk_model_metrics")
display(m[["model", "stage", "roc_auc", "precision_top10pct"]])
o = table("vis_risk_odds_ratios")
display(o[o.p_value < 0.05].round(3))
fig("vis_risk_deciles", 600); fig("vis_risk_odds_ratios", 700)

## 4 · Who buys? (customer segmentation and CLTV)

**What it does.** The loyalty-card purchases (December 2016 to February 2020) are linked to the same products and shops. IDs that buy in more than 10 shops are shop cards, not people, and are excluded. For each customer we build **RFM features** (recency, purchase days, spend per basket, items per basket). Then:
- **K-means** groups similar customers, with the number of groups chosen by **silhouette score**.
- **CLTV** (customer lifetime value) = last-12-month spend × expected lifetime, with lifetime from 12-month retention (index units, since prices are normalised).
- **Spearman correlation** tests whether frequent customers buy more short-lived or marked-down products.

**The algorithm (our code):**

In [ ]:
from src import vis_customers as CU
show(CU.purchases, "shops = c.groupby", "c = c[~c.customer.isin(cards)]")   # exclude shop cards
show(CU.customer_table)                                                     # RFM features per customer
show(CU.segment)                                                            # standardise + K-means + silhouette

In [ ]:
CU.main()   # run step 4

In [ ]:
display(table("vis_customer_segments")[["segment", "share_of_customers", "share_of_units", "share_of_spend",
                                        "markdown_share_mean", "short_lived_share_mean", "cltv_index"]])
display(table("vis_customers_by_frequency")[["freq_band", "customers", "markdown_share", "short_lived_share"]])
c = js("vis_customers_summary")
say(f"""- {c["customers"]:,} customers; the top 20% buy **{pct(c["share_units_top20pct_customers"], 0)}** of units
- Spearman, buying frequency vs markdown share: ρ = {c["spearman_frequency_vs_markdown_share"]["rho"]:.2f}; vs short-lived share: ρ = {c["spearman_frequency_vs_short_lived_share"]["rho"]:.2f}""")
fig("vis_customer_segments"); fig("vis_customers_by_frequency", 600)

## 5 · Market trends

**What it does.** From weekly loyalty-card purchases we compute a **4-week moving average**, a **linear trendline** and a **seasonal index** (each month's average week ÷ the overall average week). We also test whether **Google Trends** search interest improves the forecasts.

**The algorithm (our code):**

In [ ]:
from src import trends as T
show(T.vis_demand)
show(T.gt_value)
T.main()   # run step 5

In [ ]:
tr = js("trends_summary")
gt = tr["google_trends"]
display(table("vis_seasonal_index"))
import calendar
say(f"""- Peak month: **{calendar.month_name[tr["seasonal_peak_month"]]}** (index {tr["seasonal_peak_index"]:.2f}); low month: {calendar.month_name[tr["seasonal_trough_month"]]} (index {tr["seasonal_trough_index"]:.2f})
- June-December 2019 vs 2018: {pct(tr["yoy_growth_jun_dec"], 0)} (includes customers joining the card)
- Google Trends, before launch: WAPE {pct(gt["prelaunch_total_sales"]["with Google Trends"]["wape"])} with vs {pct(gt["prelaunch_total_sales"]["without Google Trends"]["wape"])} without; at week 2: {pct(gt["week2_weekly_wape"]["with Google Trends"])} vs {pct(gt["week2_weekly_wape"]["without Google Trends"])} → **no improvement**""")
fig("vis_demand_trend")

## 6 · How to prove it in real shops: pilot design (power analysis)

**What it does.** We plan a randomised pilot: half the flagged products follow the week-2 rule and half don't. A **power analysis** (two-sample t-test, 80% power, 5% significance) tells us how many products, or shops, each group needs.

In [ ]:
from src import pilot_design as P
show(P.main, "power = TTestIndPower()", "shops_per_arm_ancova")
P.main()   # run step 6
display(table("pilot_sample_size"))
fig("pilot_power", 600)

## 7 · Financial statements built from the data

**What it does.** Nuna Lie publishes no accounts, so we build season statements from the data itself: revenue after discounts, **markdown cost**, the value of **unsold stock**, **inventory turnover**, **days of inventory** and **gross margin**. Prices are normalised, so money is in index units (the ratios are unaffected). Unit cost is not in the data, so margins use an assumed cost of 40%, 50% or 60% of full price. We then cost the week-2 rule.

In [ ]:
from src import vis_financials as F
show(F.statement)
F.main()   # run step 7

In [ ]:
display(table("vis_financial_statement")[["season", "sell_through", "markdown_rate", "stock_to_sales", "inventory_turnover_12w",
                                          "days_of_inventory", "gross_margin_cost50", "margin_after_unsold_cost50"]])
fs = js("vis_financials_summary")
say(f"""- Week-2 rule, 2019 seasons: revenue {pct(fs["policy_revenue_change"])}, gross profit after unsold stock {", ".join(f"{pct(v)} (cost {k[4:]}%)" for k, v in fs["policy_gross_profit_change"].items())}
- Gross profit rises whenever unit cost is above **{pct(fs["breakeven_cost_to_price"])}** of full price""")
fig("vis_financial_statement")

## 8 · Consumer survey: conjoint analysis and shopper groups

**What it does.** We use the real responses to our Google Form.
- **Conjoint analysis**: each person rated 8 garment options that differ in price, style, lifespan and a sustainability label. An **OLS regression on ratings demeaned within each person** (respondent fixed effects, cluster-robust errors) gives each feature's effect. Dividing by the price effect gives **willingness to pay**, with a **bootstrap** confidence interval.
- **K-means** groups respondents by their purchase motives.
- **Spearman correlation**: does trend or discount buying go with shopping frequency or with age? A group-level comparison with Nuna Lie's customers uses the same frequency bands.

**The algorithm (our code):**

In [ ]:
from src import survey_analysis as SA
show(SA.conjoint)
show(SA.bridge_with_vis)
desc = SA.run()   # run step 8

In [ ]:
display(table("survey_conjoint_partworths"))
say(f"""- Importance: {", ".join(f"{k} {pct(v, 0)}" for k, v in desc["conjoint_importance"].items())}
- Willingness to pay for a 3+ year lifespan: Rs {desc["wtp_rs"]["long_life"]:,.0f} (95% CI {desc["wtp_rs_ci95"]["long_life"][0]:,.0f}-{desc["wtp_rs_ci95"]["long_life"][1]:,.0f}); stated preference, values above the Rs 700 tested gap are extrapolation
- Trend influence vs buying frequency: ρ = {desc["rank_corr_frequency_vs_trend"]["rho"]:.2f} (p = {desc["rank_corr_frequency_vs_trend"]["p_value"]:.3f}); vs age: ρ = {desc["rank_corr_age_vs_trend"]["rho"]:.2f} (p = {desc["rank_corr_age_vs_trend"]["p_value"]:.3f})""")
display(table("survey_segments")[["name", "respondents", "Trend", "Discount", "Novelty", "Quality", "Environment", "keep_years"]])
display(table("survey_vis_bridge_by_frequency"))
for n in ("survey_lifetime", "survey_disposal", "survey_attitudes", "survey_conjoint_importance"):
    fig(n)

## Summary: the headline numbers

Every result above is also written to `reports/claims_register.csv`. These are the numbers in our slides.

In [ ]:
claims = pd.read_csv("reports/claims_register.csv").set_index("claim_id")
keys = ["vis_leftover_share", "lc_peak_first3", "lc_first_md_week", "lc_md_after_half", "risk_auc_product", "risk_auc_plan_gb",
        "risk_auc_2wk", "vis_fc_wape", "vis_fc_naive_wape", "cust_top_segment", "cust_md_lo_hi", "sim_leftover_change",
        "sim_revenue_change", "sim_tonnes", "fin_markdown_rate", "fin_stock_to_sales", "fin_policy_gp_cost50",
        "sv_n", "sv_keep_mean", "sv_imp_lifespan"]
display(claims.loc[[k for k in keys if k in claims.index], ["claim", "value"]])
print(len(claims), "results recorded in total")